# MP1 - SmallBank (A)
**Corey Farrow** · Comm3220 DM4DM

Server `f-sg6m-s4.comm.virginia.edu`, database `SmallBankDB` (T-SQL).

Formatting rules from the assignment: money with `$`, commas and no decimals (`FORMAT(x,'C0')`); integers with commas (`'N0'`); interest rates with three decimals (`'N3'`). `FORMAT` returns text, so every `ORDER BY` sorts on the original numeric column.

**Q1.** Produce an alphabetized list of our customers who live in Dallas. Sort by last name, then first name.

In [ ]:
-- No columns specified, so show them all.
SELECT *
FROM CUSTOMER
WHERE city = 'Dallas'
ORDER BY l_name, f_name;

**Q2.** Show all the loans above $200K, print out all the info we have on them from that table. Order by principal, descending.

In [ ]:
-- "All the info ... from that table" overrides the formatting bullet, so SELECT *.
SELECT *
FROM LOAN
WHERE principal > 200000
ORDER BY principal DESC;

**Q3.** How many loans below $75K do we have in the DB?

In [ ]:
SELECT FORMAT(COUNT(*), 'N0') AS 'Loans Below $75K'
FROM LOAN
WHERE principal < 75000;

**Q4.** Show id, rate, and principal for our ten smallest loans. Smallest on top. Tip: check the results!

In [ ]:
-- "Check the results": rows with a missing principal would sort first in ascending
-- order and are not real loans, so exclude them.
SELECT TOP 10
    l_id                      AS 'Loan ID',
    FORMAT(rate, 'N3')        AS 'Interest Rate (%)',
    FORMAT(principal, 'C0')   AS 'Principal'
FROM LOAN
WHERE principal IS NOT NULL
ORDER BY principal ASC;

**Q5.** What is our total exposure for non-expired loans as of today? (exposure = sum of borrowed principals; today = date in which we run the query).

In [ ]:
-- GETDATE() returns the date the query runs; a loan is not expired if it is due today or later.
SELECT FORMAT(SUM(principal), 'C0') AS 'Total Exposure (Non-Expired Loans)'
FROM LOAN
WHERE date_due >= CAST(GETDATE() AS DATE);

**Q6.** What is the average loan size at the 5% rate?

In [ ]:
-- Rates are stored as percent numbers (the handout filters on 7.5 and 8.5), so 5% = 5.
SELECT FORMAT(AVG(principal), 'C0') AS 'Average Loan Size at 5%'
FROM LOAN
WHERE rate = 5;

**Q7.** Show customer ID and address for N. Grover. Do not assume you know the full first name.

In [ ]:
-- Match only the first initial. CUSTOMER stores city and state as the address.
SELECT
    c_id    AS 'Customer ID',
    f_name  AS 'First Name',
    l_name  AS 'Last Name',
    city    AS 'City',
    state   AS 'State'
FROM CUSTOMER
WHERE f_name LIKE 'N%'
  AND l_name = 'Grover';

**Q8.** How many customers do we have in each city?

In [ ]:
-- Group on city AND state so same-named cities in different states stay separate.
SELECT
    city                    AS 'City',
    state                   AS 'State',
    FORMAT(COUNT(*), 'N0')  AS 'Number of Customers'
FROM CUSTOMER
GROUP BY city, state
ORDER BY COUNT(*) DESC, city;

**Q9.** What is the average loan size by rate? Order by rate, descending. Tip: check the results!

In [ ]:
-- "Check the results": a NULL rate would appear as its own group, so leave it out.
SELECT
    FORMAT(rate, 'N3')            AS 'Interest Rate (%)',
    FORMAT(AVG(principal), 'C0')  AS 'Average Loan Size'
FROM LOAN
WHERE rate IS NOT NULL
GROUP BY rate
ORDER BY rate DESC;

**Q10.** Do we have a customer who has the initials NM? (print out all the info we have in the customer table on them)

In [ ]:
SELECT *
FROM CUSTOMER
WHERE f_name LIKE 'N%'
  AND l_name LIKE 'M%';

**Q11.** How many customers last-named "Echevarria" do we have?

In [ ]:
-- Exact match, so = rather than LIKE.
SELECT FORMAT(COUNT(*), 'N0') AS 'Customers Named Echevarria'
FROM CUSTOMER
WHERE l_name = 'Echevarria';

**Q12.** What are our average rates and average principals by year of maturity? Order by year.

In [ ]:
-- YEAR() pulls the year out of the due (maturity) date.
SELECT
    YEAR(date_due)                AS 'Maturity Year',
    FORMAT(AVG(rate), 'N3')       AS 'Average Rate (%)',
    FORMAT(AVG(principal), 'C0')  AS 'Average Principal'
FROM LOAN
GROUP BY YEAR(date_due)
ORDER BY YEAR(date_due);